In [3]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [4]:
## LOAD THE TRAINED MODEL, SCALER PICKLE, ONEHOT
model = load_model('model.h5')

## LOAD THE ENCODER AND SCALER
with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo= pickle.load(file)
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender= pickle.load(file)
with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [5]:
## EXAMPLE INPUT DATA
input_data = {
    'CreditScore':600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary':50000
}

In [6]:
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\ANN_IMPLEMENTATION\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [7]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [8]:
## ENCODE CATEGORIAL VARIABLES
input_df['Gender']= label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [9]:
## CONCATINATION WITH ONE HOT ENCODED DATA
input_df=pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [10]:
###SCALING THE INPUT DATA
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.57601113,  0.92528293,  0.10829704, -0.58482277, -0.09195652,
         0.83577637,  0.67616189,  0.96559161, -0.98665992,  0.96922337,
        -0.55521021, -0.57542576]])

In [11]:
### Prediction
prediction=model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 160ms/step


array([[0.22417288]], dtype=float32)

In [12]:
precdition_proba = prediction[0][0]
precdition_proba

0.22417288

In [13]:
if precdition_proba > 0.5:
    print('The customer is likely to churn')
else:
    print("The customer is not likely to churn")

The customer is not likely to churn
